# Module 2: Download SEC Financial Data

目标：读取模块 1 的清单，先为 **AEO、ANF、URBN** 获取 Company Facts，保存原始 JSON，查看财务字段和存货记录。
本文件可在新的 Kernel 中独立运行，不依赖模块 1 的内存变量。代码使用普通循环、字典和基础表格操作。

**沿用旧版模块 1 的 CSV 即可，不必重新做模块 1。** 两个 Notebook 必须使用同一个 `PROJECT_ROOT`。
本机只做离线模拟检查，尚未联网下载这三家公司的真实财务数据；没有预填结果。

Company Facts 每次返回一家公司的标准、公司整体层面的财务事实，包含多个标签、期间和披露版本。
它不是已经整理好的年度报表，也不涵盖所有自定义或分部标签。
本模块保留原始记录，不按 `fy` 简单分组，不把季度、年度和重复披露直接混成年度样本。
原始数据包括今天可取得的历史披露；后续预测研究需按 `filed` 日期控制信息可用时间，并处理重述。

来源：[SEC Company Facts 官方说明](https://www.sec.gov/search-filings/edgar-application-programming-interfaces)、
[SEC 访问频率说明](https://www.sec.gov/about/developer-resources)。


## Cell 1: Install Packages

和模块 1 使用相同工具。已经安装过可跳过。


In [ ]:
%pip install requests pandas


## Cell 2: Set Up the Project and Requests

确认 `PROJECT_ROOT` 与模块 1 一致，替换联系邮箱。
原始财务数据放在 `work/data/raw/companyfacts`，清单仍从 `work/data/processed` 读取。
每次新请求前暂停一秒，顺序下载三家公司；默认使用缓存。


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import time
import pandas as pd
import requests
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name in ["notebooks", "guides", "outputs"]:
    PROJECT_ROOT = PROJECT_ROOT.parent
FACTS_DIR = PROJECT_ROOT / "work" / "data" / "raw" / "companyfacts"
PROCESSED_DIR = PROJECT_ROOT / "work" / "data" / "processed"
FACTS_DIR.mkdir(parents=True, exist_ok=True)

SEC_CONTACT = "your_email@example.com"  # Replace with your contact email
REFRESH = False
if SEC_CONTACT == "your_email@example.com":
    raise ValueError("Replace the placeholder with your contact email.")
SEC_HEADERS = {"User-Agent": f"RetailInventoryResearch/0.1 (contact: {SEC_CONTACT})"}
print("Project directory:", PROJECT_ROOT)


## Cell 3: Read the Company Universe

`read_csv` 读取模块 1 的结果；`dtype` 保留 CIK 开头的零。
用 `.isin` 选出三家公司。这里只检查试点是否齐全、编号是否缺失，避免向错误地址请求数据。
如果找不到文件，请核对项目目录，不要重新手工填写 CIK。


In [ ]:
universe_path = PROCESSED_DIR / "retail_candidate_universe.csv"
universe = pd.read_csv(universe_path, dtype={"ticker": "string", "cik10": "string"})
PILOT_TICKERS = ["AEO", "ANF", "URBN"]
pilot = universe.loc[universe["ticker"].isin(PILOT_TICKERS)].copy()
if len(pilot) != 3 or pilot["cik10"].isna().any():
    raise ValueError("The pilot companies or CIKs are missing. Check Module 1.")
display(pilot[["ticker", "sec_company_name", "cik10"]])


## Cell 4: Download and Cache Company Facts

`pilot.iterrows()` 每次给出一个行索引和该行内容，`row["ticker"]` 与 `row["cik10"]` 取出公司信息。
`company_facts` 是字典：股票代码作为键，这家公司的原始财务对象作为值。

每家公司先查看 CIK 命名的缓存文件；需要下载时才发出请求。最多三次新请求，重跑默认零次。
保存来源、时间和原始内容；CIK 比对防止缓存或响应指向另一家公司。
HTTP 错误时停止当前 Cell，之前成功保存的文件仍保留。解决问题后重跑，会先使用这些缓存。
403、429 或超时后先暂停，不要连续重试；这里不设置自动重试和复杂异常包装。


In [ ]:
company_facts = {}
for index, row in pilot.iterrows():
    ticker = row["ticker"]
    cik10 = row["cik10"]
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik10}.json"
    cache_path = FACTS_DIR / f"CIK{cik10}_companyfacts.json"

    if cache_path.exists() and not REFRESH:
        snapshot = json.loads(cache_path.read_text(encoding="utf-8"))
        origin = "cache"
    else:
        time.sleep(1)
        response = requests.get(url, headers=SEC_HEADERS, timeout=30)
        if response.status_code != 200:
            raise RuntimeError(f"{ticker}: HTTP {response.status_code}. Stop and retry later.")
        snapshot = {
            "source_url": url,
            "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            "payload": response.json(),
        }
        origin = "download"

    if int(snapshot["payload"]["cik"]) != int(cik10):
        raise ValueError(f"{ticker}: CIK mismatch in the snapshot.")
    if origin == "download":
        cache_path.write_text(json.dumps(snapshot, indent=2), encoding="utf-8")
    company_facts[ticker] = snapshot["payload"]
    print(ticker, origin, snapshot["retrieved_at_utc"])


## Cell 5: Inspect One Company's Structure

先看 AEO，避免一次面对三家公司的所有内容。
`facts` 下按会计分类存放标签；`.get("us-gaap", {})` 取出 US GAAP 标签，没有时返回空字典。
`us_gaap` 的每个键（tag）表示一个财务项目。这里展示前十个标签，仅用于认识结构。
下一格会查找研究需要的标签，不需要你现在记住全部英文名称。


In [ ]:
aeo = company_facts["AEO"]
us_gaap = aeo["facts"].get("us-gaap", {})
print("Company:", aeo["entityName"])
print("US GAAP tag count:", len(us_gaap))
print("Example tags:", list(us_gaap.keys())[:10])


## Cell 6: List Candidate Financial Tags

先查五类项目：收入、销售成本、存货、总资产和经营现金流。
公司可能换用标签，因此为部分项目列出多个候选。循环只记录有哪些标签，不自动选定或合并它们。

`record_count_usd` 是这个标签下 USD 原始披露记录的数量，**不是可用财年数**。
它可能含季度、累计期间、重复比较数和重述。0 表示没有找到候选标签或该标签没有 USD 记录，不能填成财务金额 0。
费用范围（例如是否包括配送、折旧）、现金流合并范围和历史标签衔接，会在后续模块核对。
若找不到候选，不等于公司没有披露该项目；可能需要检查其他标准标签或年报自定义标签。


In [ ]:
tag_candidates = {
    "revenue": ["RevenueFromContractWithCustomerExcludingAssessedTax", "SalesRevenueNet", "Revenues", "SalesRevenueGoodsNet"],
    "cost_of_sales": ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold"],
    "inventory": ["InventoryNet"],
    "total_assets": ["Assets"],
    "operating_cash_flow": ["NetCashProvidedByUsedInOperatingActivities", "NetCashProvidedByUsedInOperatingActivitiesContinuingOperations"],
}
rows = []
for ticker, data in company_facts.items():
    tags = data["facts"].get("us-gaap", {})
    for metric, possible_tags in tag_candidates.items():
        found = False
        for tag in possible_tags:
            if tag in tags:
                records = tags[tag]["units"].get("USD", [])
                rows.append({"ticker": ticker, "metric": metric, "tag": tag, "record_count_usd": len(records)})
                found = True
        if not found:
            rows.append({"ticker": ticker, "metric": metric, "tag": None, "record_count_usd": 0})

tag_summary = pd.DataFrame(rows)
display(tag_summary)


## Cell 7: View Inventory Records and Save the Tag Summary

`InventoryNet` → `units` → `USD` 是一组原始存货披露记录。
这里只查看 AEO 的存货示例，展示按披露日期排序后的最后八条，**不是八个不同财年**。

- `end`：存货余额对应的时点；存货属于时点值，通常没有 `start`。
- `val`：金额；这里选取 USD，保留原始美元单位。
- `form`：披露文件类型，如 10-K 或 10-Q。
- `filed`：文件提交日期，决定信息何时可获得。
- `accn`：文件编号，方便追溯来源。

同一个 `end` 重复出现可能是不同文件重复披露或重述，暂不删除。来自 10-K 也不一定只含本财年的记录。
保存标签概览供下一模块参考；三家公司的完整原始数据已在 Cell 4 保存。


In [ ]:
inventory_data = us_gaap.get("InventoryNet", {}).get("units", {}).get("USD", [])
inventory = pd.DataFrame(inventory_data)
if inventory.empty:
    print("No USD InventoryNet records found for AEO. Review the annual report.")
else:
    inventory = inventory.sort_values(["filed", "end"])
    display(inventory[["end", "val", "form", "filed", "accn"]].tail(8))

summary_path = PROCESSED_DIR / "pilot_financial_tag_summary.csv"
tag_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
print("Saved tag summary:", summary_path)
print("Raw financial snapshots:", FACTS_DIR)


## Completion Checklist

完成后应有三条公司获取状态、三份 CIK 命名的 JSON 快照、一张财务标签概览，以及 AEO 的存货预览或缺失提示。
标签缺失应保留记录，不能为了得到完整表格自行填金额。

三份快照应对应 `CIK0000919012`（AEO）、`CIK0001018840`（ANF）和 `CIK0000912615`（URBN）；
实际编号读取自模块 1，不在下载逻辑中手工写死。
这一步证明已取得原始数据，不证明有多少个完整、可比的年度样本。

**下一步模块 3：** 选定和核实财务标签，区分时点值和期间值，匹配真实年度期间，处理重复披露与重述，再构造年度表。
本模块不需要扩大到 42 家，先理解三家公司的结构即可。
